In [1]:
import stim
import itertools
import numpy as np
import pickle
import time
import os
import re
import pprint
import numpy as np

from autqec.automorphisms   import *
from autqec.utils.qec       import *
from autqec.utils.qiskit    import *
from autqec.graph_auts      import *
from autqec.ZX_dualities    import *
from autqec.ZY_dualities    import *
from autqec.magma_interface import *
from autqec.code_embedding  import *

from magma_online           import *
from stim_helpers           import *
from compare_circuits       import *
from cws_helpers            import * 


### 5-qubit code

In [2]:
generators_5qubits = [
    stim.PauliString("XZZXI"),
    stim.PauliString("IXZZX"),
    stim.PauliString("XIXZZ"),
    stim.PauliString("ZZXIX")
]

# logical operators
logical_I = stim.PauliString("IIIII")
logical_X = stim.PauliString("XXXXX")
logical_Z = stim.PauliString("ZZZZZ")
logical_Y = canonicalize_real_pauli(logical_X * logical_Z)

logical_X_min = stim.PauliString("-YIXIY")
logical_Z_min = stim.PauliString("-XIZIX")
logical_Y_min = canonicalize_real_pauli(logical_X_min * logical_Z_min)

logical_basis5 = {
    "I": logical_I,
    "X": canonicalize_real_pauli(logical_X_min),
    "Y": canonicalize_real_pauli(logical_Y_min),
    "Z": canonicalize_real_pauli(logical_Z_min)
}

### [[10,2,3]] stabilizer generators: 2 independent copies

In [3]:
generators_5qubits_2blocks = []

for s in generators_5qubits:
    generators_5qubits_2blocks.append(s + logical_I)
    generators_5qubits_2blocks.append(logical_I + s)

# single-block logicals on each encoded qubit
logical_XI = logical_X_min + logical_I
logical_ZI = logical_Z_min + logical_I

logical_IX = logical_I + logical_X_min
logical_IZ = logical_I + logical_Z_min

logical_YI = canonicalize_real_pauli(logical_XI * logical_ZI)
logical_IY = canonicalize_real_pauli(logical_IX * logical_IZ)

# convenient logical basis for generators
logical_basis_2_generators = {
    "XI": canonicalize_real_pauli(logical_XI),
    "YI": canonicalize_real_pauli(logical_YI),
    "ZI": canonicalize_real_pauli(logical_ZI),

    "IX": canonicalize_real_pauli(logical_IX),
    "IY": canonicalize_real_pauli(logical_IY),
    "IZ": canonicalize_real_pauli(logical_IZ),
}

# full 2-qubit Pauli logical basis, excluding II
logical_basis_2blocks = {}

for a in ["I", "X", "Y", "Z"]:
    for b in ["I", "X", "Y", "Z"]:
        label = a + b
        if label == "II":
            continue

        pauli = logical_basis5[a] + logical_basis5[b]
        logical_basis_2blocks[label] = canonicalize_real_pauli(pauli)

print_combined_stabilizers(
    generators_5qubits_2blocks,
    "Combined generators: two blocks of [[5,1,3]] -> [[10,2,3]]",
    num_blocks = 2
)

print("Number of stabilizer generators: ", len(generators_5qubits_2blocks))
print("Number of logical basis elements:", len(logical_basis_2blocks))


Combined generators: two blocks of [[5,1,3]] -> [[10,2,3]]

+XZZXIIIIII   +IIIIIXZZXI
+IXZZXIIIII   +IIIIIIXZZX
+XIXZZIIIII   +IIIIIXIXZZ
+ZZXIXIIIII   +IIIIIZZXIX

Number of stabilizer generators:  8
Number of logical basis elements: 15


In [4]:
# Prologue / epilogue for two 5-qubit blocks

prologue = stim.Circuit()
prologue.append("I", [i for i in range(10)])
prologue.append("H", [0])
prologue.append("S", [0])
prologue.append("Y", [2])
prologue.append("H", [4])
prologue.append("S", [4])
prologue.append("H", [5])
prologue.append("S", [5])
prologue.append("Y", [7])
prologue.append("H", [9])
prologue.append("S", [9])

RR_part1 = []
RR_part1.append(("CZ", [0,5]))
RR_part1.append(("CZ", [2,7]))
RR_part1.append(("CZ", [4,9]))
RR_part1.append(("CZ", [0,9]))
RR_part1.append(("CZ", [2,5]))
RR_part1.append(("CZ", [4,7]))

RR_part2 = []
RR_part2.append(("CZ", [0,7]))
RR_part2.append(("CZ", [2,9]))
RR_part2.append(("CZ", [4,5]))

epilogue = stim.Circuit()
epilogue.append("I", [i for i in range(10)])
epilogue.append("S", [0])
epilogue.append("S", [0])
epilogue.append("S", [0])
epilogue.append("H", [0])
epilogue.append("Y", [2])
epilogue.append("S", [4])
epilogue.append("S", [4])
epilogue.append("S", [4])
epilogue.append("H", [4])
epilogue.append("S", [5])
epilogue.append("S", [5])
epilogue.append("S", [5])
epilogue.append("H", [5])
epilogue.append("Y", [7])
epilogue.append("S", [9])
epilogue.append("S", [9])
epilogue.append("S", [9])
epilogue.append("H", [9])

In [5]:
def update_logical_basis_by_circuit(logical_basis: dict, circuit: stim.Circuit) -> dict:
    """
    Conjugate every logical operator in a basis dict by a Stim circuit.
    Returns a new dict with the same keys and updated PauliStrings.
    """
    return {
        label: canonicalize_real_pauli(conjugate_stabilizer_by_circuit(op, circuit))
        for label, op in logical_basis.items()
    }

In [6]:
# code after prologue

generators_5qubits_2blocks_after_prologue = []
for s in generators_5qubits_2blocks:
    generators_5qubits_2blocks_after_prologue.append(conjugate_stabilizer_by_circuit(s, prologue))

print_combined_stabilizers(
    generators_5qubits_2blocks_after_prologue,
    "Combined generators after prologue circuit: two blocks of [[5,1,3]] -> [[10,2,3]]",
    num_blocks = 2
)

logical_basis_2blocks_after_prologue = update_logical_basis_by_circuit(logical_basis_2blocks, prologue)


Combined generators after prologue circuit: two blocks of [[5,1,3]] -> [[10,2,3]]

-ZZZXIIIIII   -IIIIIZZZXI
-IXZZZIIIII   -IIIIIIXZZZ
-ZIXZYIIIII   -IIIIIZIXZY
-YZXIZIIIII   -IIIIIYZXIZ



### Converting to CWS Codes

In [7]:
graph_stabs, C, A, hadamard_qubits = stabilizer_code_to_cws(
    stabilizers = generators_5qubits_2blocks_after_prologue,
    logical_Zs  = [logical_basis_2blocks_after_prologue["IZ"], logical_basis_2blocks_after_prologue["ZI"]],
    logical_Xs  = [logical_basis_2blocks_after_prologue["IX"], logical_basis_2blocks_after_prologue["XI"]],
)

print("Graph stabilizers after prologue:")
print("Standard Form:")
for g in graph_stabs:
    print(g)

print("Code Words =", C)
for c in C:
    print(c)

print("Adj Matrix =\n", A)

Graph stabilizers after prologue:
Standard Form:
+XZ_ZZ_____
+ZX_Z______
+__XZZ_____
+ZZZX______
+Z_Z_X_____
+_____XZ_ZZ
+_____ZX_Z_
+_______XZZ
+_____ZZZX_
+_____Z_Z_X
Code Words = ['0000000000', '0000001001', '0100100000', '0100101001']
0000000000
0000001001
0100100000
0100101001
Adj Matrix =
 [[0 1 0 1 1 0 0 0 0 0]
 [1 0 0 1 0 0 0 0 0 0]
 [0 0 0 1 1 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 1 1]
 [0 0 0 0 0 1 0 0 1 0]
 [0 0 0 0 0 0 0 0 1 1]
 [0 0 0 0 0 1 1 1 0 0]
 [0 0 0 0 0 1 0 1 0 0]]


In [8]:
def apply_cz_to_cws(C, A, gate_list):
    """
    Apply CZ gates to a CWS code.

    CZ is diagonal, so C is unchanged. In graph-state language,
    CZ(a,b) toggles the ordinary graph edge {a,b}.
    """
    n = A.shape[0]

    edges = {}
    for i in range(n):
        for j in range(i + 1, n):
            if A[i, j]:
                edges[frozenset({i, j})] = 1

    for gate, qubits in gate_list:
        if gate != "CZ":
            raise ValueError(f"Unsupported gate: {gate}. Only CZ is supported.")
        if len(qubits) != 2:
            raise ValueError(f"CZ expects 2 qubits, got {qubits}.")

        key = frozenset(qubits)
        edges[key] = 1 - edges.get(key, 0)
        if edges[key] == 0:
            del edges[key]

    return list(C), edges


def edges_to_adj(edges, n):
    A_new = np.zeros((n, n), dtype=np.uint8)
    for e in edges:
        if len(e) != 2:
            continue
        i, j = tuple(e)
        A_new[i, j] = 1
        A_new[j, i] = 1
    return A_new


def print_graph_edges(edges):
    edges_2 = sorted([sorted(e) for e in edges if len(e) == 2])
    print(f"  2-edges ({len(edges_2)}): {edges_2}")

In [9]:
# Apply part 1 of the round robin CZ layer
C_after_part1, edges_after_part1 = apply_cz_to_cws(C, A, RR_part1)
A_after_part1 = edges_to_adj(edges_after_part1, n=10)

print("Updated codewords after part 1:")
for c in C_after_part1:
    print(c)

print("\nGraph after part 1:")
print_graph_edges(edges_after_part1)

Updated codewords after part 1:
0000000000
0000001001
0100100000
0100101001

Graph after part 1:
  2-edges (18): [[0, 1], [0, 3], [0, 4], [0, 5], [0, 9], [1, 3], [2, 3], [2, 4], [2, 5], [2, 7], [4, 7], [4, 9], [5, 6], [5, 8], [5, 9], [6, 8], [7, 8], [7, 9]]


In [10]:
# Apply part 2 as a second diagonal CZ layer
C_after_part2, edges_after_part2 = apply_cz_to_cws(C_after_part1, A_after_part1, RR_part2)
A_after_part2 = edges_to_adj(edges_after_part2, n=10)

print("Updated codewords after part 2:")
for c in C_after_part2:
    print(c)

print("\nGraph after part 2:")
print_graph_edges(edges_after_part2)

Updated codewords after part 2:
0000000000
0000001001
0100100000
0100101001

Graph after part 2:
  2-edges (21): [[0, 1], [0, 3], [0, 4], [0, 5], [0, 7], [0, 9], [1, 3], [2, 3], [2, 4], [2, 5], [2, 7], [2, 9], [4, 5], [4, 7], [4, 9], [5, 6], [5, 8], [5, 9], [6, 8], [7, 8], [7, 9]]


### Classical-code automorphisms

In [11]:
n = 10
k = 2
d = 3

command_file_name = f"./magma_commands_n{n}k{k}d{d}.txt"
output_file_name  = f"./magma_output_n{n}k{k}d{d}.txt"

# Use the two single-logical-qubit generator codewords.
# With the logical_Xs order above, these are C[1] and C[2].
C_new = C_after_part1
C_new_gens = [C_new[1], C_new[2]]

magma_lines = []
magma_lines.append("F := GF(2);")
magma_lines.append("G := Matrix(F, [")
magma_lines += [
    "  [" + ",".join(row) + "]" + ("," if i < len(C_new_gens) - 1 else "")
    for i, row in enumerate(C_new_gens)
]
magma_lines.append("]);")
magma_lines.append("C := LinearCode(G);")
magma_lines.append("A := AutomorphismGroup(C);")
magma_lines.append("A;")
magma_lines.append("Generators(A);")
magma_lines.append("#A;")

with open(output_file_name, "w") as output_file:
    pass

with open(command_file_name, "w") as f:
    f.write("\n".join(magma_lines))

run_magma_online_from_file(command_file_name)

with open(output_file_name, "r") as f:
    magma_output = f.read()

print(magma_output)


def parse_magma_permutation_group(output_text, n):
    """
    Parse Magma permutation generators into Python 0-indexed permutations of length n.
    Skips the set literal { ... } block that Magma prints for Generators(A).
    """
    cleaned = re.sub(r'\{[^}]*\}', '', output_text, flags=re.DOTALL)

    perms = []
    for line in cleaned.splitlines():
        line = line.strip()
        if not line or not line.startswith('('):
            continue

        perm = list(range(n))
        cycles = re.findall(r'\(([^)]+)\)', line)
        for cyc in cycles:
            cyc_nums = [int(x) - 1 for x in cyc.split(',')]
            for i in range(len(cyc_nums)):
                perm[cyc_nums[i]] = cyc_nums[(i + 1) % len(cyc_nums)]
        perms.append(perm)

    return perms


auts = parse_magma_permutation_group(magma_output, n=n)
print("num generators =", len(auts))
for p in auts:
    print(p)

Permutation group A acting on a set of cardinality 10
Order = 5760 = 2^7 * 3^2 * 5
(3, 4)
(2, 7)(5, 10)
(4, 6)
(2, 5)
(1, 9)
(1, 3)
(6, 8)
(7, 10)
{
(7, 10),
(3, 4),
(4, 6),
(2, 7)(5, 10),
(2, 5),
(1, 9),
(1, 3),
(6, 8)
}
5760

num generators = 8
[0, 1, 3, 2, 4, 5, 6, 7, 8, 9]
[0, 6, 2, 3, 9, 5, 1, 7, 8, 4]
[0, 1, 2, 5, 4, 3, 6, 7, 8, 9]
[0, 4, 2, 3, 1, 5, 6, 7, 8, 9]
[8, 1, 2, 3, 4, 5, 6, 7, 0, 9]
[2, 1, 0, 3, 4, 5, 6, 7, 8, 9]
[0, 1, 2, 3, 4, 7, 6, 5, 8, 9]
[0, 1, 2, 3, 4, 5, 9, 7, 8, 6]


In [12]:
def check_codeword_preserving(perm, C_new):
    """Check if a permutation maps the classical code to itself."""
    C_set = set(C_new)
    for word in C_new:
        permuted = "".join(word[perm[i]] for i in range(len(perm)))
        if permuted not in C_set:
            return False
    return True


def check_edge_preserving(perm, edges):
    """Check if a permutation maps every graph edge to another graph edge."""
    edge_set = set(edges.keys()) if isinstance(edges, dict) else set(edges)
    for edge in edge_set:
        permuted_edge = frozenset(perm[q] for q in edge)
        if permuted_edge not in edge_set:
            return False
    return True


def filter_graph_automorphisms(auts, C_new, edges):
    valid = []
    for i, p in enumerate(auts):
        preserves_C = check_codeword_preserving(p, C_new)
        preserves_G = check_edge_preserving(p, edges)
        if preserves_C and preserves_G:
            print(f"gen {i:2d}: preserves C={preserves_C}  preserves G={preserves_G}")
            valid.append((i, p))
    return valid


valid_auts = filter_graph_automorphisms(auts, C_after_part1, edges_after_part1)
print(f"\n{len(valid_auts)} / {len(auts)} generators preserve both C and graph after part 1:")
for i, p in valid_auts:
    print(f"  gen {i:2d}: {p}")


0 / 8 generators preserve both C and graph after part 1:


In [13]:
def compose(p, q):
    # p after q: i -> q[i] -> p[q[i]]
    return [p[q[i]] for i in range(len(p))]


def invert_perm(p):
    inv = [0] * len(p)
    for i, j in enumerate(p):
        inv[j] = i
    return inv


def closure_limited(gens, n, max_size=200000):
    ident = tuple(range(n))
    seen = {ident}
    frontier = [list(ident)]
    gens2 = gens + [invert_perm(g) for g in gens]

    while frontier and len(seen) < max_size:
        cur = frontier.pop(0)
        for g in gens2:
            nxt = tuple(compose(g, cur))
            if nxt not in seen:
                seen.add(nxt)
                frontier.append(list(nxt))

    return [list(p) for p in seen]


auts_closure = closure_limited(auts, n=10, max_size=100000)

for name, C_piece, edges_piece in [
    ("part1", C_after_part1, edges_after_part1),
    ("part2", C_after_part2, edges_after_part2),
]:
    valid = [
        p for p in auts_closure
        if check_codeword_preserving(p, C_piece) and check_edge_preserving(p, edges_piece)
    ]

    print(name, "valid =", len(valid))
    for p in valid[:20]:
        print(p)

part1 valid = 1
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
part2 valid = 2
[5, 6, 7, 8, 9, 0, 1, 2, 3, 4]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [14]:
def logical_action(perm, C_new):
    C_list = list(C_new)
    C_set  = {w: i for i, w in enumerate(C_list)}
    action = []
    for word in C_list:
        permuted = "".join(word[perm[i]] for i in range(len(perm)))
        if permuted not in C_set:
            raise ValueError(f"Permutation does not preserve codespace: {word} -> {permuted}")
        action.append(C_set[permuted])
    return action


def is_identity_action(action):
    return all(action[i] == i for i in range(len(action)))


print("Searching for graph-CWS automorphisms with nontrivial logical action after part 1...\n")

results = []
for p in auts_closure:
    if not check_codeword_preserving(p, C_after_part1):
        continue
    if not check_edge_preserving(p, edges_after_part1):
        continue

    action = logical_action(p, C_after_part1)
    trivial = is_identity_action(action)

    if not trivial:
        results.append((p, action))
        print("*** VALID NONTRIVIAL SYMMETRY FOUND ***")
        print(f"perm={p}")
        print(f"logical action={action}")
        print()

print(f"Summary: {len(results)} nontrivial graph-CWS symmetries found after part 1")

Searching for graph-CWS automorphisms with nontrivial logical action after part 1...

Summary: 0 nontrivial graph-CWS symmetries found after part 1


In [15]:
# Optional sanity check for expected C ordering in the 2-logical-qubit code.
# Depending on the CWS conversion Hadamard layer, the bitstrings can differ,
# but there should always be exactly 4 codewords.
assert len(C) == 4
assert len(C_after_part1) == 4
assert C_after_part1 == C
assert C_after_part2 == C